# FinBERT Sentiment API for ORCA upstream

Run this notebook on Kaggle to expose `ProsusAI/finbert` behind a small FastAPI service.

Output labels stay FinBERT-native (`positive`, `neutral`, `negative`). `stock_bigdata` maps them to ORCA labels (`BULLISH`, `NEUTRAL`, `BEARISH`, aggregate `MIXED`).

In [ ]:
!pip install -q fastapi uvicorn transformers torch pyngrok requests

In [ ]:
import os
import threading
from typing import Any

import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "ProsusAI/finbert"
PORT = int(os.getenv("PORT", "8000"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

app = FastAPI(title="FinBERT Sentiment API", version="0.1.0")

class SentimentRequest(BaseModel):
    text: str = Field(min_length=1)

class BatchSentimentRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=64)

def predict_text(text: str) -> dict[str, Any]:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].detach().cpu()
    scores = {str(model.config.id2label[idx]).lower(): float(probs[idx]) for idx in range(len(probs))}
    positive_prob = scores.get("positive", 0.0)
    neutral_prob = scores.get("neutral", 0.0)
    negative_prob = scores.get("negative", 0.0)
    label = max(scores, key=scores.get) if scores else "neutral"
    return {
        "label": label,
        "positive_prob": positive_prob,
        "neutral_prob": neutral_prob,
        "negative_prob": negative_prob,
        "sentiment_score": positive_prob - negative_prob,
        "scores": scores,
    }

@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok", "model": MODEL_ID, "device": str(device)}

@app.post("/sentiment")
def sentiment(req: SentimentRequest) -> dict[str, Any]:
    return predict_text(req.text)

@app.post("/sentiment/batch")
def sentiment_batch(req: BatchSentimentRequest) -> dict[str, Any]:
    return {"results": [predict_text(text) for text in req.texts]}

## Optional ngrok tunnel

Set Kaggle secret/env `NGROK_AUTHTOKEN` first. If not set, skip this cell and use another tunnel/proxy.

In [ ]:
NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN")
if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    public_url = ngrok.connect(PORT, "http").public_url
    print(f"FINBERT_API_URL={public_url}")
else:
    print("NGROK_AUTHTOKEN missing; tunnel not started")

In [ ]:
thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info"),
    daemon=True,
)
thread.start()
print(f"Serving on 0.0.0.0:{PORT}")

In [ ]:
import requests

response = requests.post(
    f"http://127.0.0.1:{PORT}/sentiment",
    json={"text": "Apple shares rose after stronger than expected earnings guidance."},
    timeout=30,
)
response.raise_for_status()
response.json()